In [5]:
# Cell 1: Data Acquisition (Corrected)
# ============================================================================
import os
import re
import requests
import gzip
import shutil

# ----------------------------------------------------------------------------
# 1. Define the 8 strains (RefSeq assembly accessions)
# ----------------------------------------------------------------------------
STRAINS = {
    "K12_MG1655": "GCF_000005845.2",
    "O157_H7_Sakai": "GCF_000008865.2",
    "O104_H4": "GCF_000017325.1",
    "O111_H": "GCF_000026585.1",
    "O26_H11": "GCF_000026605.1",
    "O103_H2": "GCF_000010745.1",
    "O145_H28": "GCF_000520035.1",
    "HS_commensal": "GCF_000012945.1"
}

# ----------------------------------------------------------------------------
# 2. Create directory structure
# ----------------------------------------------------------------------------
RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
RESULTS_DIR = "data/results"
FIGURES_DIR = "figures"

for d in [RAW_DIR, PROCESSED_DIR, RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

print("[OK] Directory structure verified.")

# ----------------------------------------------------------------------------
# 3. Resolve the real NCBI FTP directory for each accession
# ----------------------------------------------------------------------------
# NCBI's layout is:
#   https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/
# The "ASM584v2" part (assembly name) is assigned independently by NCBI and
# CANNOT be reliably derived from the accession digits, so instead of
# guessing it we list the remote directory and read the real folder name.

BASE_URL = "https://ftp.ncbi.nlm.nih.gov/genomes/all/"

def get_remote_dir_url(accession):
    """
    Build the directory URL that CONTAINS the assembly-specific subfolder,
    e.g. https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/
    """
    prefix, acc_num_full = accession.split("_")       # "GCF", "000005845.2"
    acc_num = acc_num_full.split(".")[0]               # "000005845"
    folder = "/".join([acc_num[:3], acc_num[3:6], acc_num[6:9]])  # "000/005/845"
    return f"{BASE_URL}{prefix}/{folder}/"

def resolve_assembly_folder(accession):
    """
    Fetch the parent directory listing and find the actual subfolder name
    that starts with the accession (e.g. GCF_000005845.2_ASM584v2).
    Returns the full URL to that assembly's folder (with trailing slash),
    or None if it can't be resolved.
    """
    dir_url = get_remote_dir_url(accession)
    try:
        resp = requests.get(dir_url, timeout=30)
        resp.raise_for_status()
    except Exception as e:
        print(f"  [ERROR] Could not list directory {dir_url}: {e}")
        return None

    matches = re.findall(r'href="(' + re.escape(accession) + r'[^"/]*)/"', resp.text)
    if not matches:
        print(f"  [ERROR] No matching assembly folder found under {dir_url}")
        return None

    folder_name = matches[0]
    return dir_url + folder_name + "/", folder_name

def download_file(url, dest_path):
    """Download with progress indicator."""
    if os.path.exists(dest_path):
        print(f"  [SKIP] {os.path.basename(dest_path)} already exists.")
        return True
    try:
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()
        with open(dest_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"  [OK] Downloaded {os.path.basename(dest_path)}")
        return True
    except Exception as e:
        print(f"  [ERROR] Failed to download {url}: {e}")
        return False

print("\n[INFO] Starting downloads...\n")
for name, acc in STRAINS.items():
    print(f"[{name}] Resolving remote folder for {acc}...")
    resolved = resolve_assembly_folder(acc)
    if resolved is None:
        print(f"  [ERROR] Skipping {name} ({acc}) — could not resolve assembly folder.\n")
        continue

    assembly_url, folder_name = resolved

    # Genome file (.fna.gz)
    fna_url = f"{assembly_url}{folder_name}_genomic.fna.gz"
    fna_dest = os.path.join(RAW_DIR, f"{name}.fna.gz")
    download_file(fna_url, fna_dest)

    # Annotation file (.gff.gz)
    gff_url = f"{assembly_url}{folder_name}_genomic.gff.gz"
    gff_dest = os.path.join(RAW_DIR, f"{name}.gff.gz")
    download_file(gff_url, gff_dest)
    print()

# ----------------------------------------------------------------------------
# 4. Decompress files
# ----------------------------------------------------------------------------
print("[INFO] Decompressing files...")
for filename in os.listdir(RAW_DIR):
    if filename.endswith(".gz"):
        gz_path = os.path.join(RAW_DIR, filename)
        out_path = gz_path[:-3]

        if os.path.exists(out_path):
            continue

        with gzip.open(gz_path, 'rb') as f_in:
            with open(out_path, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        print(f"  [OK] Decompressed {filename}")

# ----------------------------------------------------------------------------
# 5. Write manifest
# ----------------------------------------------------------------------------
manifest_path = os.path.join(RAW_DIR, "sources.txt")
with open(manifest_path, 'w') as f:
    f.write("# E. coli strains used in the pan-genome pipeline\n")
    f.write("# Name\tAccession\tSource\n")
    f.write("K12_MG1655\tGCF_000005845.2\tLab strain (K-12 substr. MG1655)\n")
    f.write("O157_H7_Sakai\tGCF_000008865.2\tEnterohemorrhagic pathogen\n")
    f.write("O104_H4\tGCF_000017325.1\tOutbreak strain (Germany 2011)\n")
    f.write("O111_H\tGCF_000026585.1\tEnteropathogenic pathogen\n")
    f.write("O26_H11\tGCF_000026605.1\tEnterohemorrhagic pathogen\n")
    f.write("O103_H2\tGCF_000010745.1\tEnterohemorrhagic pathogen (str. 12009)\n")
    f.write("O145_H28\tGCF_000520035.1\tEnterohemorrhagic pathogen (str. RM13514)\n")
    f.write("HS_commensal\tGCF_000012945.1\tCommensal isolate from healthy human\n")

print(f"\n[OK] Manifest saved to {manifest_path}")

# ----------------------------------------------------------------------------
# 6. Verification
# ----------------------------------------------------------------------------
fna_files = [f for f in os.listdir(RAW_DIR) if f.endswith(".fna") and not f.endswith(".gz")]
gff_files = [f for f in os.listdir(RAW_DIR) if f.endswith(".gff") and not f.endswith(".gz")]

print("\n[VERIFICATION] Downloaded and decompressed:")
print(f"  Genome files (.fna): {len(fna_files)} / {len(STRAINS)}")
print(f"  Annotation files (.gff): {len(gff_files)} / {len(STRAINS)}")

if len(fna_files) == len(STRAINS) and len(gff_files) == len(STRAINS):
    print("\n[SUCCESS] All files are ready for the pipeline.")
else:
    print("\n[WARNING] Some files are missing. Check the downloads above.")

[OK] Directory structure verified.

[INFO] Starting downloads...

[K12_MG1655] Resolving remote folder for GCF_000005845.2...
  [SKIP] K12_MG1655.fna.gz already exists.
  [SKIP] K12_MG1655.gff.gz already exists.

[O157_H7_Sakai] Resolving remote folder for GCF_000008865.2...
  [SKIP] O157_H7_Sakai.fna.gz already exists.
  [SKIP] O157_H7_Sakai.gff.gz already exists.

[O104_H4] Resolving remote folder for GCF_000017325.1...
  [SKIP] O104_H4.fna.gz already exists.
  [SKIP] O104_H4.gff.gz already exists.

[O111_H] Resolving remote folder for GCF_000026585.1...
  [SKIP] O111_H.fna.gz already exists.
  [SKIP] O111_H.gff.gz already exists.

[O26_H11] Resolving remote folder for GCF_000026605.1...
  [SKIP] O26_H11.fna.gz already exists.
  [SKIP] O26_H11.gff.gz already exists.

[O103_H2] Resolving remote folder for GCF_000010745.1...
  [OK] Downloaded O103_H2.fna.gz
  [OK] Downloaded O103_H2.gff.gz

[O145_H28] Resolving remote folder for GCF_000520035.1...
  [OK] Downloaded O145_H28.fna.gz
  [O